# 03 - Agente RAG: base normativa, recuperação, avaliação e logs

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
fig = lambda n: display(Image(filename=str(ROOT / 'reports/figures' / n)))
met = lambda n: pd.read_csv(ROOT / 'reports/metrics' / n)

In [2]:
import collections
from src.rag.retrieve import load_chunks
ch = load_chunks(); print(len(ch), 'chunks')
pd.Series(collections.Counter(c['sigla'] for c in ch)).rename('chunks').to_frame()

452 chunks


,chunks
Resolução CONAMA 491/2018,35
Resolução CONAMA 506/2024,31
Resolução CONAMA 03/1990,14
Resolução CONAMA 05/1989,18
Lei 6.938/1981,183
Decreto 99.274/1990,71
Lei 9.605/1998,100


Exemplo de chunk (unidade = artigo, com dispositivos citáveis):

In [3]:
c = next(c for c in ch if c['doc_id']=='conama_491_2018' and c['label'].startswith('art.')); print(json.dumps({k: c[k] for k in ['id','sigla','path','label','dispositivos','vigencia_inicio','vigencia_fim']}, ensure_ascii=False, indent=1)); print(c['text'][:1200])

{
 "id": "conama_491_2018:art1",
 "sigla": "Resolução CONAMA 491/2018",
 "path": "",
 "label": "art. 1º",
 "dispositivos": [
  "art. 1º"
 ],
 "vigencia_inicio": "2018-11-21",
 "vigencia_fim": "2024-07-08"
}
Art. 1º Esta Resolução estabelece padrões de qualidade do ar.


## Benchmark de recuperação (recall@k, MRR)

In [4]:
met('rag_retrieval.csv').round(3)

,config,recall@1,recall@3,recall@5,recall@10,hit@1,hit@3,hit@5,hit@10,MRR,latencia_ms,n,indexacao_s
0,BM25,0.617,0.850,0.917,1.000,0.700,0.900,0.933,1.0,0.797,7.598,30,NaN
1,Vetorial (embeddinggemma-300m),0.683,0.850,0.883,0.967,0.767,0.900,0.933,1.0,0.838,162.346,30,1.667
2,Híbrido RRF (BM25 + embeddinggemma-300m),0.650,0.850,0.950,0.983,0.733,0.900,0.967,1.0,0.832,168.939,30,NaN
3,Vetorial (granite-multilingual-278m),0.583,0.750,0.900,0.983,0.667,0.800,0.933,1.0,0.764,116.180,30,3.880
4,Híbrido RRF (BM25 + granite-multilingual-278m),0.717,0.833,0.983,1.000,0.800,0.900,1.000,1.0,0.864,138.907,30,NaN
5,Vetorial (qwen3-emb-0.6b),0.500,0.733,0.867,0.867,0.533,0.800,0.900,0.9,0.662,1360.651,30,1.624
6,Híbrido RRF (BM25 + qwen3-emb-0.6b),0.600,0.833,0.917,0.983,0.667,0.867,0.933,1.0,0.782,1410.210,30,NaN
7,Híbrido RRF (BM25 + granite-multilingual-278m)...,0.717,0.817,0.933,0.983,0.800,0.900,1.000,1.0,0.864,132.679,30,NaN
8,"Híbrido + reranker Qwen3-0.6B (top 8, granite-...",0.700,0.867,0.950,1.000,0.767,0.967,1.000,1.0,0.864,61485.535,30,NaN


Limiar de recusa: maior escore do reranker por pergunta (respondíveis x sem resposta na base)

In [5]:
lim = met('rag_limiar_recusa.csv'); lim = lim[lim['modo']=='hybrid_rerank']
lim.assign(sem_resposta=lim['tipo'].eq('sem_resposta')).groupby('sem_resposta')['top_rerank'].describe().round(3)

,count,mean,std,min,25%,50%,75%,max
sem_resposta,,,,,,,,
False,31.0,0.994,0.014,0.930,0.995,0.999,1.000,1.000
True,6.0,0.416,0.384,0.017,0.084,0.413,0.618,0.993


## Avaliação ponta a ponta (acerto, citação, fidelidade, recusa, ferramenta)

In [6]:
met('rag_eval_resumo.csv').round(3).T

,0,1,2,3
modelo,qwen3-4b-instruct-2507-q4_k_m,qwen3-4b_ferramenta_regra,qwen3-4b_ferramenta_v2,gemma3-4b-it-q4_k_m
n,37,3,3,37
acerto,0.871,1.0,1.0,0.71
acerto_parcial,0.919,1.0,1.0,0.758
cit_precisao,0.891,0.75,0.833,0.826
cit_recall,0.948,1.0,1.0,0.958
fidelidade_auto,1.0,0.667,0.667,0.96
fidelidade_juiz,1.0,NaN,0.667,0.644
recusa_correta (sem resposta),1.0,NaN,NaN,1.0
recusa_indevida,0.064,0.0,0.0,0.194


In [7]:
import glob
for f in sorted(glob.glob('reports/metrics/rag_eval_detalhe_*.csv'))[:1]:
    d = pd.read_csv(f); display(Markdown(f'`{f}`')); display(d.drop(columns=['resposta']).round(2))

`reports/metrics\rag_eval_detalhe_gemma3-4b-it-q4_k_m.csv`

,id,tipo,encontrado,tempo_s,tokens,acerto,acerto_parcial,recusa_indevida,cit_precisao,cit_recall,fidelidade_auto,fidelidade_juiz,juiz_sem_suporte,ferramenta_ok,recusa_correta
0,q01,fato,True,244.59,2058,True,1.0,False,0.50,1.0,1.0,0.5,O padrão legal vigente na data da estimativa (...,NaN,NaN
1,q02,fato,True,194.65,1996,True,1.0,False,0.50,1.0,1.0,0.3,"O valor do padrão intermediário PI-1 para MP2,...",NaN,NaN
2,q03,fato,True,212.18,2180,True,1.0,False,1.00,1.0,1.0,1.0,NaN,NaN,NaN
3,q04,fato,True,166.98,1915,True,1.0,False,1.00,1.0,1.0,0.5,O prazo foi até 2024-07-08,NaN,NaN
4,q05,fato,True,218.35,2307,True,1.0,False,1.00,1.0,1.0,0.5,O nível de atenção para episódio crítico de po...,NaN,NaN
5,q06,fato,True,238.35,2777,True,1.0,False,1.00,1.0,1.0,0.5,A data de vigência da Resolução 491/2018 foi d...,NaN,NaN
6,q07,fato,True,184.94,1677,True,1.0,False,1.00,1.0,1.0,1.0,NaN,NaN,NaN
7,q08,fato,True,194.51,2004,True,1.0,False,1.00,1.0,1.0,1.0,NaN,NaN,NaN
8,q09,fato,True,355.95,4018,True,1.0,False,1.00,1.0,1.0,1.0,NaN,NaN,NaN
9,q10,fato,True,282.25,2344,True,1.0,False,0.50,1.0,1.0,0.7,A legislação também prevê a possibilidade de p...,NaN,NaN


## Exemplo de execução registrada (pergunta, trechos, ferramentas, resposta, tempo, custo)

In [8]:
runs = [json.loads(l) for l in open('logs/agent_runs.jsonl', encoding='utf-8')]
r = next((r for r in runs if any(t['ferramenta']=='estimar_pm25' for t in r['ferramentas'])), runs[-1])
print(r['pergunta']); print(); print(r['resposta']); print()
print(json.dumps({k: r[k] for k in ['tempo_total_s','tokens_entrada','tokens_saida','custo_estimado_usd','trechos','validacao']}, ensure_ascii=False, indent=1))

qual o PM2.5 previsto para Campinas amanhã e ele ultrapassa o padrão da Resolução CONAMA 491?

O PM2.5 previsto para Campinas amanhã (2023-04-05) é de 20,7 µg/m³, com intervalo de 90% entre 11,7 e 29,1 µg/m³. O padrão de qualidade do ar vigente na data de referência (2023-04-04) é o da Resolução CONAMA 491/2018, art. 4º, § 1º e Anexo I, com o valor de referência para o período de 24 horas no PI-1 de 60 µg/m³ [1][6]. O valor previsto (20,7 µg/m³) é menor que 60 µg/m³, portanto, não ultrapassa o padrão legal. A Resolução CONAMA 491/2018 foi revogada por meio da Resolução CONAMA 506/2024, art. 11, inciso I, mas a revogação entrou em vigor em 2024-07-08, ou seja, na data de referência (2023-04-04), a Resolução 491/2018 ainda era vigente [1]. Assim, a avaliação se baseia no padrão vigente na data.

Fontes:
[1] Resolução CONAMA 491/2018, Anexo I (Material Particulado - MP2,5, 24 horas) — https://faolex.fao.org/docs/pdf/bra215752.pdf
[6] Resolução CONAMA 506/2024, Anexo I (Material Particulad